# Chapter 9, Part 3. Reaction rates, equilibrium, and selectivity

A favorable reaction can be slow, and the product that forms first need not dominate at equilibrium. This notebook connects free-energy barriers to rate constants, concentration changes, and competing products.

**All free-energy parameters below are invented teaching values.** A, B, P, and Q are abstract isomeric states with the same molecular composition. These are internally consistent kinetic models, not calculated or measured molecular predictions.

### Learning objectives

- Distinguish an electronic barrier, an activation Gibbs energy, and an Arrhenius activation energy.
- Apply the Eyring equation with correct unimolecular and bimolecular units.
- Check that reversible rate constants reproduce a defined equilibrium constant.
- Verify a numerical kinetic solution against conservation and an analytic solution.
- Explain how catalysis and observation time influence rates and selectivity.

**Setup:** run independently in the course environment from the [README](Readme.md). The notebook uses NumPy, SciPy, pandas, and Matplotlib; all calculations and plots are local and short. [Part 1](Chapter09_Part1.ipynb) introduces reaction thermodynamics, and [Part 2](Chapter09_Part2.ipynb) discusses transition structures and mechanisms.

### Start here: amount, speed, and time are different quantities

Concentration $[A]$ is an amount per volume. A reaction **rate** is a change of concentration per time. A **rate constant** connects the rate to concentrations through a stated rate law; for a first-order elementary step, $v=k[A]$ and $k$ has units s$^{-1}$.

An energy diagram describes a possible bottleneck. A kinetic equation describes how populations change when molecules repeatedly cross that bottleneck in both directions. Thermodynamics constrains the equilibrium ratio, whereas barriers set how quickly that ratio is approached.

**First pass:** follow the barrier-sensitivity table, reversible concentration curves, and product-yield decision. The analytic solution and detailed-balance identities are useful checks when later models become larger. Every rate and concentration prediction below follows from **invented teaching parameters**, not fitted data for a named chemical reaction.

## 9.3.1. Three quantities that should not share one label

| Quantity | Definition or meaning | What it does not provide alone |
|---|---|---|
| Electronic barrier $\Delta E_{\mathrm{el}}^\ddagger$ | Difference between transition-structure and reactant clamped-nuclei energies, including nuclear repulsion | Entropy, thermal effects, or a rate constant |
| Activation Gibbs energy $\Delta G^{\ddagger\circ}(T)$ | Standard activation free energy, conventionally $\Delta H^{\ddagger\circ}-T\Delta S^{\ddagger\circ}$ | A universal, temperature-independent electronic barrier |
| Arrhenius activation energy $E_a(T)$ | Temperature sensitivity: $E_a=-R\,d\ln(k/k_{\rm ref})/d(1/T)$, with a fixed unit reference $k_{\rm ref}$ | Necessarily the height of a potential-energy saddle |

For a unimolecular Eyring model with temperature-independent transmission factor and locally constant activation enthalpy/entropy, $E_a=\Delta H^{\ddagger\circ}+RT$. This is not an identity between $E_a$ and $\Delta G^{\ddagger\circ}$. More complicated mechanisms or temperature-dependent corrections can change the relation. See the [IUPAC Green Book, chemical kinetics quantities](https://iupac.qmul.ac.uk/bibliog/GreenBook3rd2pt2008.pdf).

A transition-state partition function treats the unstable reaction coordinate separately; one must not insert its imaginary frequency as an ordinary stable harmonic oscillator. Solvent, conformers, standard states, recrossing, and tunneling may matter in a real rate prediction. Here those complications are replaced by explicitly stated illustrative free-energy parameters.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import constants
from scipy.integrate import solve_ivp
from IPython.display import display

OUTPUT_DIR = Path("outputs/chapter09_part3").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
T_K = 298.15
R_J_mol_K = constants.R
C_STANDARD_M = 1.0  # Standard concentration: 1 mol/L; M means mol/L.
KAPPA = 1.0
versions = {name: version(name) for name in ("numpy", "scipy", "pandas", "matplotlib")}
print("Illustrative model only; temperature = 298.15 K; standard concentration = 1 M")
print("Package versions:", versions)

## 9.3.2. Eyring rates and their sensitivity to a barrier

For a **unimolecular** elementary step under the stated transition-state model,

$$k_1=\kappa\frac{k_BT}{h}\exp\!\left[-\frac{\Delta G^{\ddagger\circ}}{RT}\right],
\qquad v=k_1[A].$$

$k_BT/h$ has units $s^{-1}$. The activation free energy and $RT$ must use the same **molar energy units**, so the code converts kJ/mol to J/mol. The dimensionless transmission factor $\kappa$ represents a dynamical correction; we set it to 1 throughout. See [Eyring's original rate-theory paper](https://doi.org/10.1063/1.1749604) and [IUPAC transition-state theory](https://goldbook.iupac.org/terms/view/T06470/pdf).

A small error in a barrier can produce a large error in a predicted rate. At this temperature, a barrier change of $RT\ln10$ changes the rate by a factor of ten, for an unchanged prefactor. The table varies barriers at **one temperature**; it is not a temperature-dependence fit.

In [ ]:
def eyring_unimolecular(barrier_kj_mol, temperature_K=T_K, kappa=KAPPA):
    """Return k in s^-1 for a specified unimolecular activation Gibbs energy."""
    if not np.isfinite(temperature_K) or not np.isfinite(kappa) or temperature_K <= 0 or kappa <= 0:
        raise ValueError("Temperature and transmission factor must be positive and finite.")
    barrier_j_mol = np.asarray(barrier_kj_mol, dtype=float) * 1000
    if not np.isfinite(barrier_j_mol).all():
        raise ValueError("Activation free energy must be finite.")
    return kappa * constants.k * temperature_K / constants.h * np.exp(
        -barrier_j_mol / (R_J_mol_K * temperature_K)
    )

barriers = np.array([60.0, 65.0, 70.0, 75.0])
rates = eyring_unimolecular(barriers)
rate_sensitivity = pd.DataFrame({"illustrative_barrier_kJ_mol": barriers, "k_s-1": rates,
                                 "irreversible_first_order_half_life_s": np.log(2) / rates})
display(rate_sensitivity.round(6))
tenfold_barrier_kj_mol = R_J_mol_K * T_K * np.log(10) / 1000
assert np.isclose(eyring_unimolecular(65 + tenfold_barrier_kj_mol) / eyring_unimolecular(65), 0.1)
print(f"A +{tenfold_barrier_kj_mol:.3f} kJ/mol barrier change makes k ten times smaller.")

## 9.3.3. Build a reversible model from one shared transition state

Consider a closed, ideal dilute mixture at constant temperature and volume:

$$A\ \mathop{\rightleftharpoons}^{k_f}_{k_r}\ B.$$

Use standard free energies $G_A^\circ=0$, $G_B^\circ=-5$, and a common transition-state level $G_\ddagger^\circ=65$ kJ/mol. The forward barrier is 65 kJ/mol; the reverse barrier is **70**, measured from B. Both directions use the same prefactor and transmission factor.

$$\frac{k_f}{k_r}=\exp\!\left[-\frac{G_B^\circ-G_A^\circ}{RT}\right]=K^\circ,
\qquad K^\circ=\frac{a_B}{a_A}=\frac{[B]_{\mathrm{eq}}}{[A]_{\mathrm{eq}}}.$$

The concentration standard cancels for this 1-to-1 ideal isomerization. This equality enforces **detailed balance**: opposing fluxes match at equilibrium. Picking the two rate constants independently could contradict the assigned thermodynamics. The following diagram connects assigned state levels; it is not a computed minimum-energy path or a sampled free-energy surface.

In [ ]:
state_free_energies = {"A": 0.0, "B": -5.0, "TS": 65.0}  # Invented standard kJ/mol.
k_forward = float(eyring_unimolecular(state_free_energies["TS"] - state_free_energies["A"]))
k_reverse = float(eyring_unimolecular(state_free_energies["TS"] - state_free_energies["B"]))
K_standard = float(np.exp(-(state_free_energies["B"] - state_free_energies["A"]) * 1000 / (R_J_mol_K * T_K)))
assert np.isclose(k_forward / k_reverse, K_standard, rtol=1e-12)
print(f"kf = {k_forward:.4f} s^-1; kr = {k_reverse:.4f} s^-1; dimensionless K = {K_standard:.4f}")

fig, axis = plt.subplots(figsize=(7, 4), constrained_layout=True)
levels = [state_free_energies[name] for name in ("A", "TS", "B")]
axis.plot([0, 1, 2], levels, "--", color="0.55", lw=1.2)
for x, energy, label in zip([0, 1, 2], levels, ["A", "Transition-state level", "B"]):
    axis.hlines(energy, x - 0.16, x + 0.16, color="#28788e", lw=3)
    axis.text(x, energy + 3, f"{label}\n{energy:g} kJ/mol", ha="center", va="bottom")
axis.set(xlim=(-0.45, 2.45), ylim=(-15, 85), xticks=[],
         xlabel="Schematic state order; not a geometric reaction coordinate",
         ylabel="Assigned standard G relative to A (kJ/mol)", title="Illustrative reversible isomerization")
fig.savefig(OUTPUT_DIR / "assigned_free_energy_levels.png", dpi=140)
plt.show()

## 9.3.4. Concentration changes: solve, then verify

The elementary mass-action equations are

$$\frac{d[A]}{dt}=-k_f[A]+k_r[B],\qquad
\frac{d[B]}{dt}=k_f[A]-k_r[B].$$

They conserve $C_{\mathrm{tot}}=[A]+[B]$. Their analytic solution is

$$[B](t)=[B]_{\mathrm{eq}}+\big([B]_0-[B]_{\mathrm{eq}}\big)e^{-(k_f+k_r)t},
\qquad [B]_{\mathrm{eq}}=C_{\mathrm{tot}}\frac{K^\circ}{1+K^\circ}.$$

The relaxation time is $\tau=1/(k_f+k_r)$. It is not the irreversible half-life $\ln2/k_f$ listed earlier. Even at equilibrium, forward and reverse events continue; **net** conversion is zero.

We integrate both concentrations using [SciPy `solve_ivp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html), check its success, and compare every returned point with the analytic solution. The integration tolerances control numerical error, not the accuracy of the invented chemistry.

In [ ]:
C_TOTAL_M = 1e-3
initial_concentrations = np.array([C_TOTAL_M, 0.0])
tau_s = 1 / (k_forward + k_reverse)
time_s = np.linspace(0, 8 * tau_s, 401)
B_equilibrium_M = C_TOTAL_M * K_standard / (1 + K_standard)
A_equilibrium_M = C_TOTAL_M - B_equilibrium_M

def reversible_rhs(time, concentrations):
    A, B = concentrations
    net_flux = k_forward * A - k_reverse * B
    return [-net_flux, net_flux]

solution = solve_ivp(reversible_rhs, (time_s[0], time_s[-1]), initial_concentrations,
                     t_eval=time_s, method="DOP853", rtol=1e-10, atol=1e-13)
assert solution.success, solution.message
B_analytic = B_equilibrium_M + (initial_concentrations[1] - B_equilibrium_M) * np.exp(-time_s / tau_s)
analytic_concentrations = np.vstack([C_TOTAL_M - B_analytic, B_analytic])
np.testing.assert_allclose(solution.y, analytic_concentrations, rtol=1e-8, atol=2e-12)
np.testing.assert_allclose(solution.y.sum(axis=0), C_TOTAL_M, rtol=0, atol=1e-12)
assert solution.y.min() >= -1e-12
assert np.isclose(k_forward * A_equilibrium_M, k_reverse * B_equilibrium_M, rtol=1e-12)
print(f"Relaxation time: {tau_s:.5f} s; equilibrium B fraction: {B_equilibrium_M / C_TOTAL_M:.5f}")
print(f"Largest numerical-vs-analytic error: {np.max(np.abs(solution.y - analytic_concentrations)):.2e} M")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
axes[0].plot(time_s, solution.y[0] * 1000, label="A", color="#ba6035")
axes[0].plot(time_s, solution.y[1] * 1000, label="B", color="#28788e")
axes[0].plot(time_s[::20], B_analytic[::20] * 1000, "o", mfc="none", color="black", label="Analytic B")
axes[0].axhline(B_equilibrium_M * 1000, color="#28788e", ls=":", label="B at equilibrium")
axes[0].set(xlabel="Time (s)", ylabel="Concentration (mM)", title="Conserved total concentration: 1 mM")
axes[0].legend(fontsize=8)
axes[1].plot(time_s, k_forward * solution.y[0] * 1000, label="Forward flux", color="#ba6035")
axes[1].plot(time_s, k_reverse * solution.y[1] * 1000, label="Reverse flux", color="#28788e")
axes[1].set(xlabel="Time (s)", ylabel="One-way flux (mM/s)", title="Equal opposing fluxes at equilibrium")
axes[1].legend()
for axis in axes:
    axis.grid(alpha=0.25)
    axis.set_ylim(bottom=0)
fig.savefig(OUTPUT_DIR / "reversible_kinetics.png", dpi=140)
plt.show()

## 9.3.5. A catalyst changes the route, not this equilibrium

A catalyst provides an additional route and is regenerated. At fixed temperature and unchanged reactant/product standard states, it does not change $\Delta_rG^\circ$ or $K^\circ$. See [IUPAC's catalyst definition](https://goldbook.iupac.org/terms/view/C00876).

For a **schematic effective catalytic route at fixed catalyst activity**, assign a shared transition-state level 10 kJ/mol below the original one. Add its forward and reverse rates to the existing route. Both routes must obey the same equilibrium ratio. This is not an explicit catalytic mechanism or a prediction of turnover rates.

Lowering only the forward barrier while keeping the reverse rate fixed would change $k_f/k_r$ and contradict the unchanged state free energies. Actual catalyst-dependent rate laws may require catalyst binding, intermediates, and mass balances; those details are outside this two-state model.

In [ ]:
catalytic_TS_kj_mol = state_free_energies["TS"] - 10.0
extra_forward = float(eyring_unimolecular(catalytic_TS_kj_mol - state_free_energies["A"]))
extra_reverse = float(eyring_unimolecular(catalytic_TS_kj_mol - state_free_energies["B"]))
k_forward_cat = k_forward + extra_forward
k_reverse_cat = k_reverse + extra_reverse
assert np.isclose(extra_forward / extra_reverse, K_standard, rtol=1e-12)
assert np.isclose(k_forward_cat / k_reverse_cat, K_standard, rtol=1e-12)
tau_cat_s = 1 / (k_forward_cat + k_reverse_cat)
B_catalytic = B_equilibrium_M * (1 - np.exp(-time_s / tau_cat_s))
catalyst_table = pd.DataFrame({"routes": ["Original", "Original + effective catalyst route"],
                               "kf_s-1": [k_forward, k_forward_cat],
                               "kr_s-1": [k_reverse, k_reverse_cat],
                               "relaxation_time_s": [tau_s, tau_cat_s],
                               "dimensionless_K": [K_standard, k_forward_cat / k_reverse_cat]})
display(catalyst_table.round(6))
fig, axis = plt.subplots(figsize=(7, 3.8), constrained_layout=True)
axis.plot(time_s, B_analytic / C_TOTAL_M, label="Original route")
axis.plot(time_s, B_catalytic / C_TOTAL_M, label="With effective catalyst route")
axis.axhline(B_equilibrium_M / C_TOTAL_M, color="black", ls=":", label="Same equilibrium")
axis.set(xlabel="Time (s)", ylabel="Fraction in B", ylim=(0, 1), title="Faster equilibration, unchanged endpoint")
axis.legend()
axis.grid(alpha=0.25)
fig.savefig(OUTPUT_DIR / "catalytic_route.png", dpi=140)
plt.show()

## 9.3.6. A bimolecular rate constant has different units

For ideal dilute **association of distinct species**, $D+E\rightleftharpoons F$, use activities $a_i=[i]/c^\circ$ and a consistent concentration-standard activation free energy. The forward Eyring coefficient is

$$k_2=\frac{\kappa k_BT}{h\,c^\circ}\exp\!\left[-\frac{\Delta G_f^{\ddagger\circ}}{RT}\right],
\qquad v_f=k_2[D][E].$$

If concentrations are in M, $k_2$ has units **$M^{-1}$ $s^{-1}$**. The reverse unimolecular coefficient $k_{-1}$ has units $s^{-1}$, so

$$K^\circ=\frac{a_F}{a_Da_E}=\frac{c^\circ[F]_{\rm eq}}{[D]_{\rm eq}[E]_{\rm eq}}
=\frac{k_2c^\circ}{k_{-1}}.$$

The factor $c^\circ$ makes $K^\circ$ dimensionless. Setting its numerical value to 1 does not remove its units from the rate coefficient. Activation free energies must be converted when **changing the physical standard state**, as discussed in [Part 1](Chapter09_Part1.ipynb); changing only concentration units leaves the physical standard state unchanged. See [IUPAC's activation-free-energy convention](https://old.goldbook.iupac.org/html/G/G02631.html).

The following independent association example reuses the numerical barriers 65 and 70 kJ/mol only to illustrate the units. It is not the unimolecular A/B mechanism.

In [ ]:
bimolecular_forward_barrier = 65.0
bimolecular_reverse_barrier = 70.0
k2_M_inv_s = float(eyring_unimolecular(bimolecular_forward_barrier)) / C_STANDARD_M
k_minus1_s = float(eyring_unimolecular(bimolecular_reverse_barrier))
K_association = float(np.exp(5.0 * 1000 / (R_J_mol_K * T_K)))
assert np.isclose(k2_M_inv_s * C_STANDARD_M / k_minus1_s, K_association)

# Same physical standard state expressed in mol/m^3: 1 M = 1000 mol/m^3.
c_standard_mol_m3 = C_STANDARD_M * 1000
k2_m3_mol_s = float(eyring_unimolecular(bimolecular_forward_barrier)) / c_standard_mol_m3
D_M, E_M = 1e-3, 1e-2
forward_flux_M_s = k2_M_inv_s * D_M * E_M
forward_flux_mol_m3_s = k2_m3_mol_s * (1000 * D_M) * (1000 * E_M)
assert np.isclose(forward_flux_M_s, forward_flux_mol_m3_s / 1000)
print(f"k2 = {k2_M_inv_s:.4f} M^-1 s^-1 = {k2_m3_mol_s:.6f} m^3 mol^-1 s^-1")
print(f"Forward flux at [D]=1 mM, [E]=10 mM: {forward_flux_M_s:.6g} M/s")
print(f"K = k2*c_standard/k_minus1 = {K_association:.4f} (dimensionless)")

## 9.3.7. Competing products: the observation time matters

Now consider a different closed network, $P\rightleftharpoons A\rightleftharpoons Q$, with no direct P/Q step. All three are abstract isomeric states. Assign $G_A^\circ=0$, $G_P^\circ=-2$, $G_Q^\circ=-8$ kJ/mol, and shared transition-state levels of 65 kJ/mol for A/P and 69 kJ/mol for A/Q.

- Initially, with only A present, P forms faster because its forward activation barrier is lower. The **initial** product-flux ratio is $v_P/v_Q=k_{AP}/k_{AQ}$.
- Reverse reactions allow redistribution. At equilibrium, $[Q]/[P]=\exp[-(G_Q^\circ-G_P^\circ)/(RT)]$, so Q is favored.
- If the reverse reactions were negligible over the observation interval, the parallel first-order irreversible model would give $[P]/[Q]=k_{AP}/k_{AQ}$ throughout product formation. In this reversible model that ratio changes with time.

These are kinetic and thermodynamic control of **product composition**, not universal labels attached to a temperature. Time, reversibility, and the available paths matter. See [IUPAC kinetic control](https://goldbook.iupac.org/terms/view/K03398). The numerical matrix below represents the three mass-action equations; each column sums to zero, expressing conservation of $[A]+[P]+[Q]$.

In [ ]:
branch_G = {"A": 0.0, "P": -2.0, "Q": -8.0, "TS_AP": 65.0, "TS_AQ": 69.0}
k_AP = float(eyring_unimolecular(branch_G["TS_AP"] - branch_G["A"]))
k_PA = float(eyring_unimolecular(branch_G["TS_AP"] - branch_G["P"]))
k_AQ = float(eyring_unimolecular(branch_G["TS_AQ"] - branch_G["A"]))
k_QA = float(eyring_unimolecular(branch_G["TS_AQ"] - branch_G["Q"]))
rate_matrix = np.array([[-k_AP-k_AQ, k_PA, k_QA],
                        [k_AP, -k_PA, 0], [k_AQ, 0, -k_QA]])
np.testing.assert_allclose(rate_matrix.sum(axis=0), 0, atol=1e-12)
weights = np.exp(-np.array([branch_G[name] for name in ("A", "P", "Q")]) * 1000 / (R_J_mol_K * T_K))
branch_equilibrium_M = C_TOTAL_M * weights / weights.sum()
np.testing.assert_allclose(rate_matrix @ branch_equilibrium_M, 0, atol=1e-12)
assert np.isclose(k_AP * branch_equilibrium_M[0], k_PA * branch_equilibrium_M[1])
assert np.isclose(k_AQ * branch_equilibrium_M[0], k_QA * branch_equilibrium_M[2])
# Two decaying modes determine approach to equilibrium; the conserved mode has eigenvalue zero.
eigenvalues = np.linalg.eigvals(rate_matrix)
assert np.max(np.abs(eigenvalues.imag)) < 1e-10
relaxation_rates = -eigenvalues.real[eigenvalues.real < -1e-9]
assert len(relaxation_rates) == 2
branch_time_s = np.r_[0.0, np.geomspace(1e-6, 12 / relaxation_rates.min(), 501)]
branch_solution = solve_ivp(lambda t, concentrations: rate_matrix @ concentrations,
                            (0, branch_time_s[-1]), [C_TOTAL_M, 0, 0], t_eval=branch_time_s,
                            method="DOP853", rtol=1e-10, atol=1e-13)
assert branch_solution.success, branch_solution.message
np.testing.assert_allclose(branch_solution.y.sum(axis=0), C_TOTAL_M, atol=1e-12, rtol=0)
assert branch_solution.y.min() >= -1e-12
np.testing.assert_allclose(branch_solution.y[:, -1], branch_equilibrium_M, atol=1e-8, rtol=0)
kinetic_P_fraction = k_AP / (k_AP + k_AQ)
equilibrium_P_product_fraction = branch_equilibrium_M[1] / branch_equilibrium_M[1:].sum()
print(f"Initial P fraction among product fluxes: {kinetic_P_fraction:.4f}")
print(f"Equilibrium P fraction among products: {equilibrium_P_product_fraction:.4f}")
print("Equilibrium A/P/Q fractions:", np.round(branch_equilibrium_M / C_TOTAL_M, 5))

In [ ]:
product_total = branch_solution.y[1] + branch_solution.y[2]
# At t=0 no products exist, so their composition ratio is undefined; omit that point.
formed_products = product_total > 1e-12
P_product_fraction = branch_solution.y[1, formed_products] / product_total[formed_products]
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for row, label, color in [(0, "A", "#6d6d6d"), (1, "P", "#ba6035"), (2, "Q", "#28788e")]:
    axes[0].plot(branch_time_s[1:], branch_solution.y[row, 1:] / C_TOTAL_M, label=label, color=color)
axes[0].set(xscale="log", xlabel="Time (s)", ylabel="Fraction of total A + P + Q",
            ylim=(0, 1), title="An initially favored product can decline")
axes[0].legend()
axes[1].plot(branch_time_s[formed_products], P_product_fraction, color="#ba6035", label="P / (P + Q)")
axes[1].axhline(kinetic_P_fraction, color="0.35", ls="--", label="Initial kinetic limit")
axes[1].axhline(equilibrium_P_product_fraction, color="#28788e", ls=":", label="Equilibrium limit")
axes[1].set(xscale="log", xlabel="Time (s)", ylabel="P fraction among products only",
            ylim=(0, 1), title="Selectivity differs from conversion")
axes[1].legend(fontsize=8)
for axis in axes:
    axis.grid(alpha=0.25)
fig.savefig(OUTPUT_DIR / "time_dependent_selectivity.png", dpi=140)
plt.show()

### Interpreting this model without overclaiming

A high early fraction of P among products does not imply a high P yield: at very short times almost all material is still A. Product composition, reactant conversion, and isolated yield answer different questions.

A catalyst could change the relative access to P and Q and hence their **finite-time** proportions. In a closed, equilibrating system with the same thermodynamic species and conditions, changing kinetic paths alone cannot change the final equilibrium constants. Persistent driving, product removal, or changed chemical states requires a different model.

For a real reaction, obtaining defensible rate constants requires a verified mechanism and suitable free-energy/dynamical treatment. A plausible energy diagram or a single transition structure does not establish the entire reaction network. The conservation, detailed-balance, and unit checks here remain useful even when the chemistry becomes more complex.

### Worked experimental-design question: when would an ideal quench preserve most P?

Suppose P is the desired product in the illustrative $P\rightleftharpoons A\rightleftharpoons Q$ network. A high P fraction among products does not guarantee much P has been made. Reuse the existing trajectory and distinguish

$$X_A=1-[A]/C_{\rm tot},\qquad Y_P=[P]/C_{\rm tot},\qquad S_P=[P]/([P]+[Q]),\qquad Y_P=X_A S_P.$$

$X_A$ is conversion, $Y_P$ is model chemical yield, and $S_P$ is product selectivity. They are fractions, not isolated experimental yields. An **ideal instantaneous quench** is a thought experiment that freezes the composition without loss or further reaction. Locate the largest P amount among the already sampled times; no new kinetic integration or parameter fitting is needed.

In [ ]:
A_fraction, P_yield, Q_yield = branch_solution.y / C_TOTAL_M
conversion = 1 - A_fraction
selectivity = np.divide(P_yield, P_yield+Q_yield,
                        out=np.full_like(P_yield, np.nan), where=(P_yield+Q_yield)>1e-9)
valid_yield = np.isfinite(selectivity)
np.testing.assert_allclose(P_yield[valid_yield], conversion[valid_yield]*selectivity[valid_yield], atol=1e-10)
best_sample = int(np.argmax(P_yield))
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
axes[0].semilogx(branch_time_s[1:], P_yield[1:], label="P yield: P / total")
axes[0].semilogx(branch_time_s[valid_yield], selectivity[valid_yield], label="P selectivity: P / products")
axes[0].semilogx(branch_time_s[1:], conversion[1:], "--", label="A conversion")
axes[0].axvline(branch_time_s[best_sample], color="black", ls=":", label="Largest sampled P yield")
axes[0].set(xlabel="Time (s)", ylabel="Fraction", ylim=(0, 1), title="An ideal quench must balance amount and composition")
axes[0].legend(fontsize=8)
points = axes[1].scatter(conversion[valid_yield], selectivity[valid_yield], c=P_yield[valid_yield],
                         cmap="viridis", s=13, vmin=0, vmax=1)
axes[1].scatter([conversion[best_sample]], [selectivity[best_sample]], color="black", marker="*", s=120)
axes[1].set(xlabel="A conversion", ylabel="P selectivity among products", xlim=(0, 1), ylim=(0, 1),
            title="High selectivity alone can mean little product")
fig.colorbar(points, ax=axes[1], label="P yield (fraction of initial amount)")
fig.savefig(OUTPUT_DIR / "yield_selectivity_quench.png", dpi=140)
plt.show()
print(f"Largest sampled P yield: {P_yield[best_sample]:.3f} at {branch_time_s[best_sample]:.4g} s")
print(f"At that time: conversion = {conversion[best_sample]:.3f}; P selectivity = {selectivity[best_sample]:.3f}")

**Conclusion.** Waiting longer can increase conversion while decreasing the amount of P, because this reversible network redistributes products toward Q. Maximizing P's product fraction and maximizing P's amount are different objectives. The marker is a sampled maximum in this particular model, not a globally optimized laboratory reaction time.

**Next step in a real study:** estimate a defensible reaction network and rate constants from appropriate evidence, then account for sampling, quench chemistry, and recovery. **Self-check:** if conversion is 2% and P selectivity is 90%, P yield is only 1.8% before any isolation loss.

In [ ]:
rate_sensitivity.to_csv(OUTPUT_DIR / "barrier_sensitivity.csv", index=False)
catalyst_table.to_csv(OUTPUT_DIR / "catalyst_comparison.csv", index=False)
pd.DataFrame({"time_s": time_s, "A_M": solution.y[0], "B_M": solution.y[1],
              "B_analytic_M": B_analytic}).to_csv(OUTPUT_DIR / "reversible_concentrations.csv", index=False)
pd.DataFrame({"time_s": branch_time_s, "A_M": branch_solution.y[0],
              "P_M": branch_solution.y[1], "Q_M": branch_solution.y[2]}).to_csv(
    OUTPUT_DIR / "competing_product_concentrations.csv", index=False)
record = {"data_status": "Invented teaching parameters; not experimental or molecular-computation data",
          "temperature_K": T_K, "standard_concentration_M": C_STANDARD_M, "kappa": KAPPA,
          "total_concentration_M": C_TOTAL_M, "versions": versions,
          "reversible_state_G_standard_kJ_mol": state_free_energies,
          "reversible_rates_s-1": {"forward": k_forward, "reverse": k_reverse},
          "equilibrium_constant_dimensionless": K_standard,
          "additional_catalytic_TS_G_standard_kJ_mol": catalytic_TS_kj_mol,
          "association_example": {"forward_barrier_kJ_mol": bimolecular_forward_barrier,
                                  "reverse_barrier_kJ_mol": bimolecular_reverse_barrier,
                                  "k2_M-1_s-1": k2_M_inv_s, "k_minus1_s-1": k_minus1_s},
          "branch_state_G_standard_kJ_mol": branch_G,
          "branch_rates_s-1": {"A_to_P": k_AP, "P_to_A": k_PA, "A_to_Q": k_AQ, "Q_to_A": k_QA},
          "ODE_solver": {"method": "DOP853", "rtol": 1e-10, "atol_M": 1e-13}}
(OUTPUT_DIR / "model_record.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
print("Saved model definitions, concentration tables, and figures under outputs/chapter09_part3/.")

## 9.3.8. Exercises and self-checks

1. Increase both forward and reverse barriers of A/B by 5 kJ/mol by raising their shared transition-state level. What happens to each rate, the relaxation time, and $K^\circ$?
2. Keep $G_A^\circ=0$ and the original transition-state level, but change $G_B^\circ$ from -5 to -10 kJ/mol. Which rate changes? How does equilibrium shift?
3. Why does the reversible relaxation time contain $k_f+k_r$? Why does equilibrium not mean that both one-way fluxes vanish?
4. A program outputs $k_2/k_{-1}=7.5$ for an association. What units and standard concentration are needed before calling a quantity a dimensionless equilibrium constant?
5. For a unimolecular Eyring model at 298.15 K with constant $\kappa$, $\Delta H^\ddagger=50$ kJ/mol and $\Delta S^\ddagger=-50$ J/(mol K), calculate $\Delta G^\ddagger$ and the local $E_a$. Why are they different?
6. Why can P dominate the first formed products while Q dominates the final mixture? Would a formally irreversible parallel model show the same late redistribution?
7. If a catalyst accelerates only the stated forward step while the program leaves its reverse rate unchanged and claims the same equilibrium, which check fails?
8. In the branch plot, distinguish P's fraction of all molecules, P's fraction among products, and conversion of A. What evidence would be needed to predict an isolated experimental yield?

<details><summary>Answers</summary>

1. Both rates decrease by $\exp[-5000/(RT)]$; the relaxation time increases by its reciprocal. Their ratio and equilibrium composition are unchanged.
2. The forward barrier/rate stays fixed. The reverse barrier rises from 70 to 75 kJ/mol, slowing the reverse rate and increasing the equilibrium B/A ratio.
3. A positive deviation of B both increases B-to-A flux and decreases A-to-B flux. At equilibrium the two nonzero one-way fluxes cancel.
4. $k_2/k_{-1}$ has inverse-concentration units. Multiply by $c^\circ$ in matching units: $K^\circ=k_2c^\circ/k_{-1}$.
5. $\Delta G^\ddagger=64.9075$ kJ/mol; $E_a\approx\Delta H^\ddagger+RT=52.479$ kJ/mol under the stated local model. One contains the activation entropy; the other measures temperature sensitivity of the rate.
6. P's formation barrier is lower, but Q's assigned free energy is lower. Reversible paths let the products redistribute. An irreversible model with fixed first-order parallel rates retains its fixed branching ratio.
7. Detailed balance: the new $k_f/k_r$ disagrees with the assigned $K^\circ$.
8. The first denominator is A+P+Q; the second is P+Q. Starting with only A, conversion is $1-[A]/C_{\rm tot}$. Isolation also depends on competing chemistry, work-up, recovery, and the validity of the kinetic model.

</details>

Continue with [Part 4: Reaction representation and data checks](Chapter09_Part4.ipynb), or return to [Part 1: Reaction thermodynamics](Chapter09_Part1.ipynb).